# 02 · Compare the observer models

The project asks: does the **bimodal** response behaviour require an explicit
*switch* between prior and evidence, or can it *emerge* from adaptive Bayesian
*integration* once the observer learns its prior online?

The registered models bracket that question along two axes — whether the prior
strength is **fixed per block** or **learned online**, and whether the read-out
is a **selection** (switch) or a blended **integration**:

| model key | label | learns prior? | read-out |
|---|---|---|---|
| `switch` | Switch | no (fixed per block) | selection (the paper's model) |
| `basic_bayes` | Basic-Bayes | no (fixed per block) | integration (always) |
| `hb_adaptive` | HB-Adaptive | yes — α and κ | integration (integrate-after) |
| `hb_rachel` | HB-Rachel | yes — κ (α fixed) | integration (integrate-after) |
| `hb_salma` | HB-Salma | yes — geometric forget | integration (integrate-before) |
| `recombined` | Recombined | yes | integration (integrate-before) |
| `hierarchical_online` | Hier-Online | yes — mean & width | integration (mixture prior) |
| `reliability_mixture` | Reliability-Mixture | yes — reliance weight | discrete either/or mixture |

The registry (`observers/comparison/registry.py`) is the source of truth. See the
repo `README.md` and `experiments/rachel/03_model_equations/` for full descriptions.


In [ ]:
# === SETUP — run this first, every time you open a fresh Colab runtime ===
# Clones the project and makes `from observers import api` work. The installable
# package lives in the `hierarchical/` subdirectory, so we cd there before install.
import os, sys
REPO   = "NeuroMatch_2026_Behaviour"
BRANCH = "model-verification"
URL    = "https://github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git"
if "google.colab" in sys.modules:
    if not os.path.isdir(REPO):
        !git clone -q --branch {BRANCH} {URL}
    %cd {REPO}/hierarchical
    !git pull -q
    !pip install -q -e .
    print("setup complete — latest code loaded from", os.getcwd())
else:
    # Local Jupyter: assume the notebook is launched from within the checkout.
    # Walk up to the `hierarchical/` dir that contains the `observers` package.
    here = os.path.abspath(os.getcwd())
    while here != "/" and not os.path.isdir(os.path.join(here, "observers")):
        here = os.path.dirname(here)
    os.chdir(here); sys.path.insert(0, here)
    print("local run — repo root:", here, "| observers present:", os.path.isdir("observers"))


### Fit quality — AIC/BIC/CV per subject per model

One row per (model, subject), read live from the committed per-model fit folders
(`results/fits/comparison/<model>/subject<N>.json`), so the table never drifts
from the numbers. Lower is better. `cv_nll` is the held-out block cross-validation
score where available. Missing fits are simply absent, so the table honestly
reflects what has been fit.


In [ ]:
from observers import api
# Full table: model, label, subject, n_trials, k, nll, aic, bic, cv_nll
tbl = api.results_table()
tbl


### The comparison figure

**A** ΔAIC per subject across the fitted models (registry colours; a marker flags
the winner per subject) · **B** how each model *sets* the prior over trials
(mechanism, canonical parameters) · **C** the response read-out at a matched prior.
Panels B–C use illustrative parameters to show mechanism; panel A uses the real
committed AICs.


In [ ]:
api.plot_model_comparison()


### Which model wins, and does cross-validation agree?

In-sample AIC can favour more flexible models; the held-out block CV is the
honest arbiter. Below: the per-subject best model by AIC, and a check of whether
the AIC winner also wins on CV (where CV is available).

*(The older within-block switch-probability curve — `api.plot_switch_curve()` —
depended on a legacy flat fit file that the per-model reorganization removed; it
needs the online-model fits regenerated to be rebuilt.)*


In [ ]:
import pandas as pd

tbl = api.results_table()

# best model per subject by AIC (in-sample)
best_aic = (tbl.loc[tbl.groupby("subject")["aic"].idxmin(), ["subject", "label", "aic"]]
              .rename(columns={"label": "best_by_AIC"}).reset_index(drop=True))

# best model per subject by CV NLL (out-of-sample), among models that have CV
cv = tbl.dropna(subset=["cv_nll"])
best_cv = (cv.loc[cv.groupby("subject")["cv_nll"].idxmin(), ["subject", "label"]]
             .rename(columns={"label": "best_by_CV"}).reset_index(drop=True))

verdict = best_aic.merge(best_cv, on="subject", how="left")
verdict["AIC_and_CV_agree"] = verdict["best_by_AIC"] == verdict["best_by_CV"]
verdict


### Reading it honestly

- **What is fit:** six models are fit for all 12 subjects (`switch`, `basic_bayes`,
  `hb_adaptive`, `hb_rachel`, `hb_salma`, `recombined`); `hierarchical_online` and
  `reliability_mixture` are newer and only partially fit — the table shows exactly
  what exists.
- **AIC only compares within a subject.** Absolute AIC/BIC values are not
  comparable across subjects (different trial counts); compare ΔAIC *within* a
  subject, and tally per-subject wins rather than averaging raw AIC.
- **Where AIC and CV disagree, trust CV** — the held-out block cross-validation is
  the honest generalization test (it holds out whole prior-width blocks, so the
  learning dynamics are tested out-of-sample, not interpolated).

To (re)fit or cross-validate models — slow, minutes per subject; the committed
fits used multi-start:

```
!python -m observers.comparison.fit_batch --models hb_adaptive --subjects 5
!python -m observers.comparison.run_parallel --workers 3   # the whole comparison
```

Fits run in Colab are **not** saved back to GitHub (Colab's filesystem is
temporary) — push from a local checkout to keep them.
